In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import seaborn.objects as so

import sqlite3

In [ ]:
# Si no está sqlite3
#!pip install sqlite3

# Introducción a SQL en Python con SQLite

Para simplicidad vamos a usar un servidor SQL virtual (no necesitamos usuarios, password, ni ninguna complicación).

In [ ]:
# 1) Creamos una base de datos SQLite en memoria
con = sqlite3.connect(":memory:")

In [ ]:
# Creamos una tabla de productos
# No estudiamos estos comandos, los usamos para tener una tabla dentro de la base de datos
setup_sql = """
DROP TABLE IF EXISTS productos;

CREATE TABLE productos (
  producto_id INTEGER PRIMARY KEY,
  titulo      TEXT NOT NULL,
  categoria   TEXT NOT NULL,
  precio      REAL NOT NULL
);

INSERT INTO productos(producto_id, titulo, categoria, precio) VALUES
(0,'El Principito', 'libro', 12.00),
(1,'El Declive', 'libro', 18.00),
(2,'go', 'juguete',  25.00),
(3,'ajedrez', 'juguete',  15.00),
(4,'fideos', 'comida',  5.00),
(5,'pan lactal', 'comida',  8.00);

"""

con.executescript(setup_sql)

## SELECT
El comando SELECT nos permite obtener datos de una tabla.

### Sytaxis 1

```
SELECT *
FROM table;
```

Nos da todos los datos de la tabla.

In [ ]:
# Leemos toda la tabla

# En Pandas ejecutamos "queries" con read_sql_query(query string, conexión)
productos = pd.read_sql_query("SELECT * FROM productos;", con)
productos

### Sytaxis 2

```
SELECT columna1, columna2, ...
FROM table;
```
Nos da solo las las columnas indicadas.

In [ ]:
# Leemos las columnas titulo y precio
productos = pd.read_sql_query("SELECT titulo, precio FROM productos;", con)
productos

In [ ]:
# OJO! Acá no van paréntesis (ni hace falta comillas para los nombres de columnas o tablas)
productos = pd.read_sql_query("SELECT (titulo, precio) FROM productos;", con)
productos

### Sytaxis 3

```
SELECT columna1, columna2, ...
FROM table
WHERE condiciones;
```
Nos da solo las las columnas indicadas que de las filas que cumplan las condiciones.

In [ ]:
# Leemos las columnas titulo y precio de articulos que cuestan mas de 10 pesos
productos = pd.read_sql_query("SELECT titulo, precio FROM productos WHERE precio > 10;", con)
productos

In [ ]:
# Si queremos los articulos con precio entre 10 y 20...
productos = pd.read_sql_query("SELECT titulo, precio FROM productos WHERE precio > 10 AND precio < 20;", con)
productos

In [ ]:
# Acá sí podemos poner paréntesis si la condición es más compleja
# Usamos comillas simples para strings
productos = pd.read_sql_query("SELECT titulo, precio FROM productos WHERE (precio > 10) AND (categoria = 'juguete' OR categoria = 'libro');", con)
productos

## Ejercicio
Rehacemos algunos filtros de clases anteriores.
1. De la base Gapminder, seleccionar todas las filas del año 2007.
2. De la base Gapminder, seleccionar todas las filas de Argentina.
3. De la base MPG seleccionar las columnas mpg, horsepower y model_year de todos los modelos de los años 1970, 1975 y 1980.

Sugerencia para el último ítem
```
SELECT column1, column2, ...
FROM table_name
WHERE column_name IN (value1, value2, ...);
```

In [ ]:
# Cargamos los DataFrames
from gapminder import gapminder
mpg = sns.load_dataset("mpg")

In [ ]:
# Para convertir un DataFrame a SQL:
conGap = sqlite3.connect(":memory:")   # Le damos otro nombre a la conexión, representa una conexión a otra base de datos.

# Guardar el DataFrame como tabla "gapminder"
gapminder.to_sql("gapminder", conGap, index=False, if_exists="replace")

(devolvió la cantidad de filas afectadas)

In [ ]:
# Verificamos (primeras filas)
pd.read_sql_query("SELECT * FROM gapminder LIMIT 5;", conGap)

**Importante:** Esta forma de trabajar (primero cargar el DataFrame, pasarlo a SQL y despues volver a convertirlo a DataFrame) es claramente muy ineficiente. La utilizamos solo con fines **didácticos** para no montar un servidor SQL.

Tenemos que pensar que en el uso real, los datos ya van a estar en un servidor SQL.

In [ ]:
# Seleccionamos los datos del año 2007


In [ ]:
# Seleccionamos los datos de Argentina


In [ ]:
# columnas mpg, horsepower y model_year de todos los modelos de los años 1970, 1975 y 1980
conMPG = ???   # Le damos otro nombre a la conexión, representa una conexión a otra base de datos.


## Agregamos datos con GROUP BY
```
SELECT columnas_agrupacion, funcion_de_agregacion(columna)
FROM tabla
WHERE condicion
GROUP BY columna_agrupacion;
```

**Ejemplo.** Calcular la población total por continente en 2007 y graficar.

In [ ]:
# 3 comillas para texto multi-linea
pd.read_sql_query("""    
SELECT
    continent, year, SUM(pop)
FROM gapminder
WHERE year = 2007
GROUP BY continent
""", conGap)

### Para renombrar la columna
```
SELECT columnas_agrupacion, funcion_de_agregacion(columna) as nombre_columna
FROM tabla
WHERE condicion
GROUP BY columna_agrupacion;
```

In [ ]:
pd.read_sql_query("""    
SELECT
    continent, year, SUM(pop) as poblacion_total
FROM gapminder
WHERE year = 2007
GROUP BY continent
""", conGap)

In [ ]:
# Y si queremos la poblacion total en cada año?
# Funciona esto?
pob = pd.read_sql_query("""    
SELECT
    continent,
    year,
    SUM(pop) as poblacion_total
FROM gapminder
GROUP BY continent
""", conGap)

In [ ]:
pob

In [ ]:
# Cómo lo arreglamos?


### Ejercicio
Utilizando SQL generar una tabla con el total de pasajeros por año de la tabla de pasajes de avión y graficar.

In [ ]:
flights = sns.load_dataset("flights")
flights

### Ejercicio
1. Utilizando SQL generar una tabla con el peso promedio de los pingüinos por especie. Función: AVG(columna)
2. Utilizando SQL generar una tabla con el peso promedio de los pingüinos por especie e isla (es decir un valor para cada combinación de isla y especie).
3. Utilizando SQL generar una tabla con la cantidad de  pingüinos por especie e isla (es decir un valor para cada combinación de isla y especie en la que haya al menos un pingüino). Función: COUNT(*) (no contamos una columna en particular, sino filas)

In [ ]:
penguins = sns.load_dataset("penguins")
penguins

# Más adelante...
Vamos a ver cómo combinar información en distintas tablas, donde vamos a ver mejor el potencial de SQL.